# 08 — Baseline Models

**The go/no-go notebook.** Everything before this describes the data. This one asks the only question that decides whether the project is worth building out:

> **Does a model on these features beat a naive baseline?**

If it does not, no amount of jump-diffusion, calibration, or portfolio optimisation downstream will rescue it — those all sit *on top* of a forecast that has to carry signal in the first place. Better to learn that here, in an afternoon, than after weeks of building.

## The rules this notebook follows

1. **Walk-forward splits only.** Never a random shuffle. Random splits let the model train on the future and predict the past, which inflates every metric.
2. **Time-based across the whole pooled panel**, not per-ticker — otherwise one ticker's future leaks into another's training fold through a shared date.
3. **The naive baseline is always reported.** A model that cannot beat "predict no change" is not a model.
4. **Features come from `schema.py`**, so this notebook never drifts from the pipeline.

In [4]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate))
        break

from src.forecast_engine.features.schema import FEATURE_NAMES, TARGET_NAMES
from src.forecast_engine.data.loader import load_processed_features, PROCESSED_FEATURES_PATH

data = load_processed_features()
data["date"] = pd.to_datetime(data["date"])

print(f'Loaded {len(data):,} rows, {data["symbol"].nunique()} tickers, '
      f'{data["date"].min().date()} -> {data["date"].max().date()}')
print(f'Source: {PROCESSED_FEATURES_PATH}')
display(data.head())

MemoryError: 

In [ ]:
TARGET_COL = "future_return"
DIRECTION_COL = "future_direction"

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\StockForecastRisk\\notebooks\\data\\processed\\sp500_features.parquet'

## Select usable features

Any feature that is entirely missing is dropped rather than allowed to poison `dropna`.

This matters concretely: `days_to_next_earnings` is all-NaN whenever the earnings calendar is unavailable. Left in a `dropna(subset=features)`, that single column silently deletes **every row** in the dataset.

In [ ]:
available = [column for column in FEATURE_NAMES if column in data.columns]

all_missing = [column for column in available if data[column].isna().all()]
if all_missing:
    print(f"Dropping entirely-missing features: {all_missing}")

feature_cols = [column for column in available if column not in all_missing]

model_data = data.dropna(subset=feature_cols + [TARGET_COL]).reset_index(drop=True)

print(f"features used: {len(feature_cols)}")
print(f"rows after dropna: {len(model_data):,}  ({len(model_data) / len(data):.1%} of raw)")

assert len(model_data) > 0, "No rows survived dropna — check for an all-NaN feature."

## Walk-forward splits

Each fold trains on everything up to a cutoff date and tests on the window that follows. The model never sees a row dated later than the one it is predicting.

One subtlety that is easy to miss: the target looks `horizon` days into the future, so the last few training rows overlap the test window. A **purge gap** removes them. Without it, the model has partially seen its own test period.

In [ ]:
N_SPLITS = 5
PURGE_DAYS = 5  # must be >= the target horizon


def walk_forward_splits(frame, n_splits=N_SPLITS, purge_days=PURGE_DAYS):
    """Yield (train_idx, test_idx) expanding-window folds, purged at the boundary."""
    dates = np.sort(frame["date"].unique())
    fold_edges = np.array_split(dates, n_splits + 1)

    for fold in range(n_splits):
        train_end = fold_edges[fold][-1]
        test_dates = fold_edges[fold + 1]
        purge_cutoff = train_end - pd.Timedelta(days=purge_days)

        train_idx = frame.index[frame["date"] <= purge_cutoff]
        test_idx = frame.index[frame["date"].isin(test_dates)]

        if len(train_idx) and len(test_idx):
            yield train_idx, test_idx


for fold, (train_idx, test_idx) in enumerate(walk_forward_splits(model_data), start=1):
    print(
        f"fold {fold}: train {len(train_idx):>7,} rows "
        f"(to {model_data.loc[train_idx, 'date'].max().date()})  |  "
        f"test {len(test_idx):>6,} rows "
        f"({model_data.loc[test_idx, 'date'].min().date()} to "
        f"{model_data.loc[test_idx, 'date'].max().date()})"
    )

## The models

| Model | Why it is here |
|---|---|
| **Naive** (predict 0) | The floor. Beating this is the entire bar. |
| **Ridge** | A linear reference. If Ridge matches XGBoost, the relationship is linear and the extra complexity buys nothing. |
| **XGBoost** | The candidate — captures non-linearities and interactions. |

Metrics: **MAE/RMSE** for magnitude, and **directional accuracy** — which is what actually matters for trading. A model can have a poor RMSE and still be useful if it gets the *sign* right often enough.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor


def evaluate(y_true, y_pred):
    """Magnitude error plus directional accuracy (sign agreement)."""
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "dir_acc": float((np.sign(y_pred) == np.sign(y_true)).mean()),
    }


results = []

for fold, (train_idx, test_idx) in enumerate(walk_forward_splits(model_data), start=1):
    X_train = model_data.loc[train_idx, feature_cols]
    y_train = model_data.loc[train_idx, TARGET_COL]
    X_test = model_data.loc[test_idx, feature_cols]
    y_test = model_data.loc[test_idx, TARGET_COL]

    # Naive: predict no change. The bar every model must clear.
    results.append({"fold": fold, "model": "naive", **evaluate(y_test, np.zeros(len(y_test)))})

    # Ridge: scaler is fit on TRAIN ONLY — fitting on all data would leak
    # test-set statistics into training.
    scaler = StandardScaler().fit(X_train)
    ridge = Ridge(alpha=1.0).fit(scaler.transform(X_train), y_train)
    ridge_pred = ridge.predict(scaler.transform(X_test))
    results.append({"fold": fold, "model": "ridge", **evaluate(y_test, ridge_pred)})

    # XGBoost: pooled across all tickers, ticker_id/sector_id carried as features.
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        verbosity=0,
    ).fit(X_train, y_train)
    xgb_pred = xgb.predict(X_test)
    results.append({"fold": fold, "model": "xgboost", **evaluate(y_test, xgb_pred)})

    print(f"fold {fold} done")

results = pd.DataFrame(results)

## ★ The verdict

Read this table honestly.

- **`mae_vs_naive` < 0** → the model has lower error than predicting nothing. That is the minimum bar.
- **`dir_acc` > 0.50** → the model gets the direction right more often than a coin flip.

Do not squint for a win that is not there. A directional accuracy of 0.505 across five folds is noise, not signal — and one strong fold among five weak ones is variance, not skill. The consistency across folds matters as much as the average.

In [ ]:
summary = results.groupby("model")[["mae", "rmse", "dir_acc"]].mean()
naive_mae = summary.loc["naive", "mae"]

summary["mae_vs_naive"] = summary["mae"] - naive_mae
summary["beats_naive"] = summary["mae_vs_naive"] < 0

display(summary.round(6))

print("\nPer-fold directional accuracy (consistency matters as much as the mean):")
display(results.pivot(index="fold", columns="model", values="dir_acc").round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

pivot_mae = results.pivot(index="fold", columns="model", values="mae")
pivot_mae.plot(marker="o", ax=axes[0])
axes[0].set_title("MAE by fold (lower is better)")
axes[0].set_ylabel("MAE")

pivot_dir = results.pivot(index="fold", columns="model", values="dir_acc")
pivot_dir.plot(marker="o", ax=axes[1])
axes[1].axhline(0.5, color="red", linestyle="--", label="coin flip")
axes[1].set_title("Directional accuracy by fold")
axes[1].set_ylabel("accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## Feature importance

Only meaningful **if** the model beat naive. Importance from a model with no signal is just a ranking of noise.

In [ ]:
importance = (
    pd.Series(xgb.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
    .head(20)
)

importance.sort_values().plot.barh(figsize=(8, 7))
plt.title("XGBoost feature importance — final fold")
plt.tight_layout()
plt.show()

## Conclusions

Fill this in before moving on — it is the decision record for the whole project.

- **Did any model beat naive on MAE?**
- **Directional accuracy above 0.50, and consistent across folds?**
- **Did XGBoost beat Ridge?** (If not, the relationship is linear and the complexity is unearned.)
- **Which features carried the signal?**

### The decision

**If there is signal** → proceed: build the service layer, add the jump-diffusion risk engine, calibrate.

**If there is not** → stop and change something upstream, rather than building on sand. In rough order of what is worth trying:
1. A **longer horizon** — 5-day returns are close to noise; 20-day trends are more forecastable.
2. **Volatility as the target** instead of returns — volatility is genuinely more predictable than direction, and it is what the risk engine actually needs.
3. **Different features** — the current set is entirely technical; the signal may simply not be in price history.

Markets being hard to predict is the null hypothesis, not a bug in the code. A clean negative result here is a real finding and worth acting on.